# Font Identifier - Training Notebook

## Session Setup
Run the cell below first to initialize your environment, mount Drive, and pull the latest code.

In [ ]:
# ⚠️ RUN THIS FIRST after every reconnect

# ── 1. Mount Drive ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── 2. Pull latest code from GitHub ──────────────────────
import os

REPO_DIR  = '/content/fontidentifier'
DRIVE_DIR = '/content/drive/MyDrive/font-identifier'

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull   # update if already cloned
else:
    !git clone https://github.com/jtheanonymous1707-wq/fontidentifier.git {REPO_DIR}

# ── 3. Add model folder to Python path ───────────────────
import sys
MODEL_DIR = f'{REPO_DIR}/model'
if MODEL_DIR not in sys.path:
    sys.path.insert(0, MODEL_DIR)

# ── 4. Install dependencies ───────────────────────────────
!pip install -q timm supabase python-dotenv

# ── 5. Verify GPU ─────────────────────────────────────────
import torch
try:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
except:
    print("GPU not found - ensure you are using a GPU runtime!")
print(f"sys.path includes model: {MODEL_DIR in sys.path}")

# ── 6. Test import ────────────────────────────────────────
import train
print(f"train.py loaded ✅")
print(f"FontNet available: {hasattr(train, 'FontNet')}")

## Step 5 — Set Environment Variables

In [ ]:
import os
os.environ['GOOGLE_FONTS_API_KEY'] = 'your_api_key'
os.environ['SUPABASE_URL']         = 'your_supabase_url'
os.environ['SUPABASE_SERVICE_KEY'] = 'your_service_key'
print("Secrets set (locally in memory).")

## Step 6 & 7 — Extract Dataset from Drive
Instead of downloading and generating images in every session, we unzip the pre-generated files from Drive.

In [ ]:
import zipfile
import os
import glob

print("🚀 Starting extraction from Drive...")

DATA_ZIP = f'{DRIVE_DIR}/font_data.zip'

if os.path.exists(DATA_ZIP):
    print(f"📦 Found archive: {DATA_ZIP} ({os.path.getsize(DATA_ZIP)/1024/1024:.2f} MB)")
    print("⌛ Unzipping... this takes ~2-3 minutes.")
    with zipfile.ZipFile(DATA_ZIP, 'r') as z:
        # Robust extraction: handles any path structure in the zip
        z.extractall('/content/')
    print("✅ Unzip complete.")
else:
    print(f"❌ Error: {DATA_ZIP} not found! Did you upload it to the 'font-identifier' folder correctly?")

DATASET_DIR = '/content/data/dataset'
FONTS_DIR   = '/content/data/fonts'

print("\n--- Verification ---")
if os.path.exists(DATASET_DIR):
    classes = os.listdir(DATASET_DIR)
    print(f"✅ Dataset: {len(classes)} font classes found.")
else:
    print(f"❌ Dataset folder NOT found at {DATASET_DIR}")

if os.path.exists(FONTS_DIR):
    fonts = glob.glob(FONTS_DIR + '/*.*')
    print(f"✅ Fonts: {len(fonts)} files found.")
else:
    print(f"❌ Fonts folder NOT found at {FONTS_DIR}")

if not (os.path.exists(DATASET_DIR) and os.path.exists(FONTS_DIR)):
    print("\n🔍 Debug info: Listing contents of /content/data (if it exists):")
    if os.path.exists('/content/data'):
        print(os.listdir('/content/data'))
    else:
        print("/content/data does not exist.")

## Step 8 — Start Training
This loop includes the explicit auto-resume logic.

In [ ]:
import train

DRIVE_DIR = '/content/drive/MyDrive/font-identifier'

train.train(
    dataset_dir     = '/content/data/dataset',
    save_dir        = DRIVE_DIR,
    checkpoint_path = f'{DRIVE_DIR}/checkpoints/latest.pt',
    best_model_path = f'{DRIVE_DIR}/checkpoints/best_model.pt',
    epochs          = 60,
    batch_size      = 64,
)

## Step 9 — Export Final Model

In [ ]:
import os, torch, sys
sys.path.insert(0, '/content/fontidentifier/model')
import train

DRIVE_DIR       = '/content/drive/MyDrive/font-identifier'
BEST_MODEL_PATH = f'{DRIVE_DIR}/checkpoints/best_model.pt'

device = torch.device('cuda')

# Load best checkpoint
best_ckpt   = torch.load(BEST_MODEL_PATH, map_location=device)
num_classes = best_ckpt['num_classes']

print(f"Loaded best model from epoch {best_ckpt['epoch'] + 1}")
print(f"Val accuracy: {best_ckpt['val_acc']:.4f}")
print(f"Num classes:  {num_classes}")

# Rebuild model and load weights
model = train.FontNet(num_classes=num_classes).to(device)
model.load_state_dict(best_ckpt['model_state'])
model.eval()
print("Model ready for export.")

# Export as TorchScript for HF Spaces inference
model_cpu = model.cpu().eval()
scripted  = torch.jit.script(model_cpu)
scripted.save(f'{DRIVE_DIR}/font_model_scripted.pt')

size_mb = os.path.getsize(f'{DRIVE_DIR}/font_model_scripted.pt') / 1e6
print(f"✅ Model exported: {size_mb:.1f} MB")
print(f"✅ Saved to: {DRIVE_DIR}/font_model_scripted.pt")